

 **Nota:** A lo largo de este cuaderno, cualquier referencia a **v2** hace alusión al **sistema basado exclusivamente en un LLM (LLM puro)**, mientras que cualquier referencia a **v3** corresponde al **sistema híbrido**. Esta nomenclatura se mantiene porque ambas versiones fueron desarrolladas originalmente con esos identificadores y, por motivos de consistencia con el código, los experimentos y la documentación generada durante el desarrollo, dichas referencias se han conservado en este notebook.

In [10]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
import time
from pydantic import BaseModel, Field
from typing import Literal, List
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
%matplotlib inline

# Configuracion del LLM
MODELO = "mistral:7b-instruct-v0.3-q4_0"
llm = ChatOllama(
    model=MODELO,
    temperature=0,
    base_url="http://localhost:11434"
)

# Verificacion
respuesta = llm.invoke("Responde en una frase en español: ¿qué es el RSI?")
print("Conexion con Ollama correcta")
print(f"Respuesta: {respuesta.content}")

Conexion con Ollama correcta
Respuesta:  El RSI (Indicador de Oscilación Relativa de Stochastic) es un indicador técnico utilizado para determinar si los precios de un activo están sobrevaluados o subvaluados.


In [11]:
NOMBRES = {
    "AAPL": "Apple",             "MSFT": "Microsoft",        "NVDA": "NVIDIA",
    "GOOGL": "Alphabet",         "META": "Meta",              "AVGO": "Broadcom",
    "AMD": "AMD",                "ORCL": "Oracle",            "INTC": "Intel",
    "QCOM": "Qualcomm",          "JNJ": "Johnson & Johnson",  "UNH": "UnitedHealth",
    "LLY": "Eli Lilly",          "ABBV": "AbbVie",            "MRK": "Merck",
    "TMO": "Thermo Fisher",      "ABT": "Abbott",             "PFE": "Pfizer",
    "MDT": "Medtronic",          "BMY": "Bristol-Myers",      "JPM": "JPMorgan",
    "BAC": "Bank of America",    "WFC": "Wells Fargo",        "GS": "Goldman Sachs",
    "MS": "Morgan Stanley",      "BLK": "BlackRock",          "AXP": "American Express",
    "C": "Citigroup",            "USB": "U.S. Bancorp",       "PNC": "PNC Financial",
    "AMZN": "Amazon",            "TSLA": "Tesla",             "HD": "Home Depot",
    "MCD": "McDonald's",         "NKE": "Nike",               "SBUX": "Starbucks",
    "TGT": "Target",             "LOW": "Lowe's",             "F": "Ford",
    "GM": "General Motors",      "XOM": "ExxonMobil",         "CVX": "Chevron",
    "COP": "ConocoPhillips",     "SLB": "Schlumberger",       "EOG": "EOG Resources",
    "CAT": "Caterpillar",        "BA": "Boeing",              "HON": "Honeywell",
    "GE": "General Electric",    "LMT": "Lockheed Martin",
}
# Descarga de datos con persistencia local en parquet
TICKERS = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "META", "AVGO", "AMD", "ORCL", "INTC", "QCOM",
    "JNJ", "UNH", "LLY", "ABBV", "MRK", "TMO", "ABT", "PFE", "MDT", "BMY",
    "JPM", "BAC", "WFC", "GS", "MS", "BLK", "AXP", "C", "USB", "PNC",
    "AMZN", "TSLA", "HD", "MCD", "NKE", "SBUX", "TGT", "LOW", "F", "GM",
    "XOM", "CVX", "COP", "SLB", "EOG",
    "CAT", "BA", "HON", "GE", "LMT",
]

INICIO = "2019-01-01"
FIN    = "2024-12-31"
CARPETA_DATOS = "../datos"

def cargar_o_descargar(tickers, inicio, fin, carpeta):
    os.makedirs(carpeta, exist_ok=True)
    datos = {}
    descargados = 0
    cargados = 0

    for ticker in tickers:
        ruta = f"{carpeta}/{ticker}.parquet"

        if os.path.exists(ruta):
            datos[ticker] = pd.read_parquet(ruta)
            cargados += 1
        else:
            try:
                df = yf.download(ticker, start=inicio, end=fin,
                                progress=False, auto_adjust=True)
                if df.empty:
                    print(f"Sin datos: {ticker}")
                    continue
                
                # Aplanamos el multiindice ANTES de guardar
                df.columns = [col[0] for col in df.columns]
                df.to_parquet(ruta)
                datos[ticker] = df
                descargados += 1
            except Exception as e:
                print(f"Error descargando {ticker}: {e}")

    print(f"Cargados desde local : {cargados}")
    print(f"Descargados de red   : {descargados}")
    print(f"Total disponibles    : {len(datos)}/{len(tickers)}")
    return datos

datos = cargar_o_descargar(TICKERS, INICIO, FIN, CARPETA_DATOS)

print(f"\nEjemplo AAPL — ultimas 3 filas:")
print(datos["AAPL"].tail(3))

Cargados desde local : 50
Descargados de red   : 0
Total disponibles    : 50/50

Ejemplo AAPL — ultimas 3 filas:
                 Close        High         Low        Open    Volume
Date                                                                
2024-12-26  257.612732  258.686881  256.230300  256.787255  27237100
2024-12-27  254.201370  257.294489  251.685117  256.429191  42355300
2024-12-30  250.829773  252.122713  249.387654  250.859609  35557500


In [12]:
# Definicion de la estructura de salida
class SeñalInversion(BaseModel):
    ticker: str = Field(description="Simbolo de la accion en mayusculas")
    decision: Literal["BUY", "HOLD", "SELL"] = Field(
        description="Decision de inversion. SOLO puede ser BUY, HOLD o SELL en mayusculas"
    )
    confianza: int = Field(
        ge=1, le=10,
        description="Nivel de confianza del 1 al 10"
    )
    justificacion: str = Field(
        description="Justificacion breve en maximo 15 palabras en español"
    )

class SeñalBatch(BaseModel):
    señales: List[SeñalInversion] = Field(
        description="Lista de señales, una por cada ticker analizado"
    )

# LLM con structured output
llm_individual = llm.with_structured_output(SeñalInversion)
llm_batch      = llm.with_structured_output(SeñalBatch)

# Herramienta tecnica
def herramienta_tecnica(ticker: str) -> str:
    df = datos[ticker].copy()
    
    delta    = df["Close"].diff()
    ganancia = delta.clip(lower=0).rolling(14).mean()
    perdida  = (-delta.clip(upper=0)).rolling(14).mean()
    rsi      = 100 - (100 / (1 + ganancia / perdida))
    rsi_val  = round(float(rsi.iloc[-1]), 1)
    
    ema12       = df["Close"].ewm(span=12, adjust=False).mean()
    ema26       = df["Close"].ewm(span=26, adjust=False).mean()
    macd        = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    macd_val    = round(float(macd.iloc[-1]), 4)
    signal_val  = round(float(macd_signal.iloc[-1]), 4)
    
    sma20    = df["Close"].rolling(20).mean()
    std20    = df["Close"].rolling(20).std()
    bb_upper = sma20 + 2 * std20
    bb_lower = sma20 - 2 * std20
    bb_pct   = (df["Close"] - bb_lower) / (bb_upper - bb_lower)
    bb_val   = round(float(bb_pct.iloc[-1]), 2)
    
    precio = round(float(df["Close"].iloc[-1]), 2)
    
    return f"Precio:${precio} RSI:{rsi_val} MACD:{macd_val} Señal:{signal_val} BB:{bb_val}"

# Herramienta fundamental
def herramienta_fundamental(ticker: str) -> str:
    try:
        info       = yf.Ticker(ticker).info
        pe         = round(info.get("trailingPE", 0), 2)
        roe        = round(info.get("returnOnEquity", 0) * 100, 1)
        crecimiento= round(info.get("revenueGrowth", 0) * 100, 1)
        deuda      = round(info.get("debtToEquity", 0), 2)
        margen     = round(info.get("profitMargins", 0) * 100, 1)
        return f"P/E:{pe} ROE:{roe}% Crec:{crecimiento}% Deuda/Cap:{deuda} Margen:{margen}%"
    except:
        return "Datos fundamentales no disponibles"

# Herramienta de riesgo
def herramienta_riesgo(ticker: str) -> str:
    retornos   = datos[ticker]["Close"].pct_change().dropna()
    spy        = yf.download("SPY", start="2019-01-01", end="2024-12-31",
                             progress=False, auto_adjust=True)["Close"].squeeze()
    ret_spy    = spy.pct_change().dropna()
    
    vol        = round(float(retornos.std() * np.sqrt(252)) * 100, 2)
    sharpe     = round(float((retornos.mean() - 0.05/252) / retornos.std() * np.sqrt(252)), 3)
    var95      = round(float(-np.percentile(retornos, 5)) * 100, 2)
    acumulado  = (1 + retornos).cumprod()
    max_roll   = acumulado.cummax()
    drawdown   = (acumulado - max_roll) / max_roll
    max_dd     = round(float(drawdown.min()) * 100, 2)
    
    alineado   = pd.concat([retornos, ret_spy], axis=1).dropna()
    beta       = round(float(alineado.cov().iloc[0,1] / alineado.iloc[:,1].var()), 3)
    alpha      = round(float(retornos.mean()*252 - (0.05 + beta*(ret_spy.mean()*252 - 0.05))), 4)
    
    return f"Vol:{vol}% Sharpe:{sharpe} VaR95:{var95}% MaxDD:{max_dd}% Beta:{beta} Alpha:{alpha}"

# Herramienta de sentimiento
def herramienta_sentimiento(ticker: str) -> str:
    try:
        noticias = yf.Ticker(ticker).news
        if not noticias:
            return "Sin noticias recientes"
        titulares = [n.get("content", {}).get("title", "") for n in noticias[:5]]
        return " | ".join([t for t in titulares if t])
    except:
        return "Sin noticias disponibles"

# Herramienta Monte Carlo
def herramienta_montecarlo(ticker: str, dias: int = 20, simulaciones: int = 1000) -> str:
    retornos  = datos[ticker]["Close"].pct_change().dropna()
    mu        = float(retornos.mean())
    sigma     = float(retornos.std())
    precio_hoy = float(datos[ticker]["Close"].iloc[-1])
    
    dt = 1/252
    trayectorias = np.zeros((dias, simulaciones))
    trayectorias[0] = precio_hoy
    
    for t in range(1, dias):
        ruido = np.random.normal(0, 1, simulaciones)
        trayectorias[t] = trayectorias[t-1] * np.exp(
            (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * ruido
        )
    
    precios_finales  = trayectorias[-1]
    prob_subida      = round(float((precios_finales > precio_hoy).mean()) * 100, 1)
    precio_esperado  = round(float(precios_finales.mean()), 2)
    precio_p5        = round(float(np.percentile(precios_finales, 5)), 2)
    precio_p95       = round(float(np.percentile(precios_finales, 95)), 2)
    
    return (f"MonteCarlo({simulaciones} sim, {dias} dias): "
            f"Prob.subida:{prob_subida}% "
            f"Precio esperado:${precio_esperado} "
            f"Rango 90%:[${precio_p5} - ${precio_p95}]")

# Prueba de todas las herramientas con AAPL
print("Herramienta tecnica    :", herramienta_tecnica("AAPL"))
print("Herramienta fundamental:", herramienta_fundamental("AAPL"))
print("Herramienta riesgo     :", herramienta_riesgo("AAPL"))
print("Herramienta sentimiento:", herramienta_sentimiento("AAPL"))
print("Herramienta montecarlo :", herramienta_montecarlo("AAPL"))

Herramienta tecnica    : Precio:$250.83 RSI:58.4 MACD:5.645 Señal:5.6574 BB:0.63
Herramienta fundamental: P/E:34.36 ROE:141.5% Crec:16.6% Deuda/Cap:79.55 Margen:27.2%
Herramienta riesgo     : Vol:30.85% Sharpe:1.022 VaR95:2.95% MaxDD:-31.43% Beta:1.217 Alpha:0.159
Herramienta sentimiento: Microsoft’s Worst Month Since 2000: Why Is This Happening? | Why Apple’s WWDC 2026 shows a material positive reset of criticized AI strategy? | Intel Corporation (INTC)’s Stock Has Surged but Needs an Engineering Revival | Apple just billed you for the AI buildout you never asked for | Disney And Apple Almost Merged, Here's Why The Deal Didn't Work
Herramienta montecarlo : MonteCarlo(1000 sim, 20 dias): Prob.subida:50.7% Precio esperado:$250.86 Rango 90%:[$248.69 - $253.05]


In [13]:
def agente_tecnico(ticker: str) -> SeñalInversion:
    datos_tech = herramienta_tecnica(ticker)
    prompt = f"""Eres un analista tecnico experto. Analiza estos indicadores de {ticker} y da una señal:
{datos_tech}
RSI>70=sobrecompra(SELL), RSI<30=sobreventa(BUY), MACD>Señal=alcista, BB>0.8=sobrecompra, BB<0.2=sobreventa"""
    return llm_individual.invoke(prompt)

def agente_fundamental(ticker: str) -> SeñalInversion:
    datos_fund = herramienta_fundamental(ticker)
    prompt = f"""Eres un analista fundamental experto. Analiza estos ratios de {ticker} y da una señal:
{datos_fund}
P/E<20=barato(BUY), P/E>40=caro(SELL), ROE>15%=bueno, Crec>10%=bueno, Deuda/Cap<1=saludable"""
    return llm_individual.invoke(prompt)

def agente_riesgo(ticker: str) -> SeñalInversion:
    datos_risk = herramienta_riesgo(ticker)
    prompt = f"""Eres un gestor de riesgos experto. Analiza estas metricas de {ticker} y da una señal:
{datos_risk}
Sharpe>1=bueno(BUY), Sharpe<0=malo(SELL), Beta>1.5=muy volatil, Alpha>0=genera valor, MaxDD>-40%=alerta"""
    return llm_individual.invoke(prompt)

def agente_sentimiento(ticker: str) -> SeñalInversion:
    datos_sent = herramienta_sentimiento(ticker)
    prompt = f"""Eres un analista de sentimiento experto. Analiza estas noticias de {ticker} y da una señal:
{datos_sent}
Evalua si el sentimiento general es positivo(BUY), negativo(SELL) o neutro(HOLD)"""
    return llm_individual.invoke(prompt)

def agente_montecarlo(ticker: str) -> SeñalInversion:
    datos_mc = herramienta_montecarlo(ticker)
    prompt = f"""Eres un analista cuantitativo experto. Analiza esta simulacion Monte Carlo de {ticker} y da una señal:
{datos_mc}
Prob.subida>60%=BUY, Prob.subida<40%=SELL, entre 40-60%=HOLD. Considera tambien el rango de precios esperado."""
    return llm_individual.invoke(prompt)

# Prueba con AAPL
print("Probando los 5 agentes con AAPL...")
print("-"*50)

señal_tech = agente_tecnico("AAPL")
print(f"Tecnico     : {señal_tech.decision} ({señal_tech.confianza}/10) — {señal_tech.justificacion}")

señal_fund = agente_fundamental("AAPL")
print(f"Fundamental : {señal_fund.decision} ({señal_fund.confianza}/10) — {señal_fund.justificacion}")

señal_risk = agente_riesgo("AAPL")
print(f"Riesgo      : {señal_risk.decision} ({señal_risk.confianza}/10) — {señal_risk.justificacion}")

señal_sent = agente_sentimiento("AAPL")
print(f"Sentimiento : {señal_sent.decision} ({señal_sent.confianza}/10) — {señal_sent.justificacion}")

señal_mc = agente_montecarlo("AAPL")
print(f"Monte Carlo : {señal_mc.decision} ({señal_mc.confianza}/10) — {señal_mc.justificacion}")

Probando los 5 agentes con AAPL...
--------------------------------------------------
Tecnico     : HOLD (5/10) — El precio actual de AAPL ($250.83) no está en ninguna zona de compra o venta según los indicadores RSI y BB. El RSI de 58.4 indica que el mercado está ligeramente sobrevaluado, pero aún no es suficiente para generar una señal de venta (SELL). Por otro lado, el Bollinger Bands (BB) indica un nivel de volatilidad moderada con un valor de 0.63. Sin embargo, el BB no está en la zona de sobrecompra o sobreventa según los criterios establecidos (BB>0.8=sobrecompra, BB<0.2=sobreventa). El MACD de 5.645 es superior a su señal de 5.6574, lo que indica un mercado alcista. Por lo tanto, se recomienda mantener la posición actual y esperar una señal más clara antes de tomar acciones.
Fundamental : BUY (8/10) — El P/E de AAPL es menor a 20, lo que indica que la acción está considerada barata. Además, el ROE es superior al 15%, lo que significa que la empresa genera un rendimiento sobre c

In [14]:
# Perfiles de riesgo
PERFILES = {
    "conservador": {
        "descripcion": "Inversor conservador. Prioriza la preservacion del capital sobre la rentabilidad. Solo recomienda BUY si el riesgo es claramente bajo (Sharpe>1, MaxDD>-20%, VaR<3%). Prefiere HOLD ante la duda.",
        "confianza_minima": 7,
        "max_posiciones": 5,
    },
    "moderado": {
        "descripcion": "Inversor moderado. Equilibra rentabilidad y riesgo. Acepta volatilidad moderada y drawdowns hasta -30%. Recomienda BUY si los indicadores son mayoritariamente positivos.",
        "confianza_minima": 5,
        "max_posiciones": 10,
    },
    "agresivo": {
        "descripcion": "Inversor agresivo. Maximiza la rentabilidad aceptando alto riesgo. Tolera volatilidad alta y drawdowns hasta -50%. Recomienda BUY ante cualquier señal positiva.",
        "confianza_minima": 3,
        "max_posiciones": 15,
    },
}

class DecisionFinal(BaseModel):
    ticker: str = Field(description="Simbolo de la accion")
    decision: Literal["BUY", "HOLD", "SELL"] = Field(
        description="Decision final de inversion"
    )
    confianza: int = Field(ge=1, le=10, description="Confianza del 1 al 10")
    resumen_agentes: str = Field(
        description="Resumen de las señales de cada agente en una linea"
    )
    justificacion: str = Field(
        description="Justificacion final en maximo 20 palabras en español"
    )

llm_decisor = llm.with_structured_output(DecisionFinal)

def orquestador(ticker: str, perfil: str = "moderado") -> DecisionFinal:
    ticker   = ticker.upper().strip()
    contexto = PERFILES[perfil]["descripcion"]
    print(f"Analizando {ticker} [{perfil}]...")

    # Los 5 agentes analizan de forma objetiva sin conocer el perfil
    s_tech = agente_tecnico(ticker)
    s_fund = agente_fundamental(ticker)
    s_risk = agente_riesgo(ticker)
    s_sent = agente_sentimiento(ticker)
    s_mc   = agente_montecarlo(ticker)

    # Solo el orquestador conoce el perfil de riesgo
    prompt = f"""Eres un gestor de cartera experto. 
Perfil del inversor: {contexto}

Has recibido 5 analisis objetivos e independientes de {ticker}:
Agente Tecnico     : {s_tech.decision} (confianza {s_tech.confianza}/10) — {s_tech.justificacion}
Agente Fundamental : {s_fund.decision} (confianza {s_fund.confianza}/10) — {s_fund.justificacion}
Agente Riesgo      : {s_risk.decision} (confianza {s_risk.confianza}/10) — {s_risk.justificacion}
Agente Sentimiento : {s_sent.decision} (confianza {s_sent.confianza}/10) — {s_sent.justificacion}
Agente Monte Carlo : {s_mc.decision}  (confianza {s_mc.confianza}/10)  — {s_mc.justificacion}

Teniendo en cuenta el perfil del inversor, pondera las señales y toma una decision final."""

    return llm_decisor.invoke(prompt)

# Prueba comparativa con los 3 perfiles para AAPL
print("COMPARATIVA DE PERFILES — AAPL")
print("="*55)
for perfil in ["conservador", "moderado", "agresivo"]:
    r = orquestador("AAPL", perfil)
    print(f"\nPerfil {perfil.upper()}:")
    print(f"  Decision  : {r.decision}")
    print(f"  Confianza : {r.confianza}/10")
    print(f"  Motivo    : {r.justificacion}")

COMPARATIVA DE PERFILES — AAPL
Analizando AAPL [conservador]...

Perfil CONSERVADOR:
  Decision  : HOLD
  Confianza : 6/10
  Motivo    : El análisis técnico no cumple con las condiciones para una compra o venta basadas en los indicadores RSI y MACD, lo que indica un mercado estable pero sin tendencia clara. El fundamental es positivo debido a un P/E menor a 20, ROE superior al 15%, tasa de crecimiento superior a 10% y rendimiento sobre los activos propios positivo, pero hay que tener en cuenta una carga de deuda más alta que el promedio. El agente de riesgo indica un rendimiento superior al del mercado en términos de retorno sobre el riesgo tomado y una relación razonable con el mercado, pero advierte sobre un mayor riesgo debido a MaxDD (-31.43%). El agente sentimiento sugiere una compra basada en la noticia positiva de la WWDC de Apple en 2026, pero hay que tener en cuenta que la fecha es futura y cualquier predicción puede ser incierta. Finalmente, el análisis Monte Carlo indica una

In [15]:
import time

def calcular_datos_batch(ticker: str) -> str:
    try:
        tech = herramienta_tecnica(ticker)
        mc   = herramienta_montecarlo(ticker)
        return f"{ticker}: {tech} | {mc}"
    except:
        return f"{ticker}: sin datos"

def ranking_50(tickers, perfil="moderado", batch_size=5):
    todas_señales = []
    grupos = [tickers[i:i+batch_size] for i in range(0, len(tickers), batch_size)]
    contexto = PERFILES[perfil]["descripcion"]
    inicio_total = time.time()

    for idx, grupo in enumerate(grupos):
        inicio_grupo = time.time()
        print(f"Procesando grupo {idx+1}/{len(grupos)}: {grupo}", end=" ")

        datos_grupo = "\n".join([calcular_datos_batch(t) for t in grupo])
        prompt = f"""Perfil del inversor: {contexto}

Analiza estas acciones y devuelve una señal para cada una acorde al perfil:
{datos_grupo}"""

        try:
            resultado = llm_batch.invoke(prompt)
            todas_señales.extend(resultado.señales)
        except Exception as e:
            print(f"Error: {e}")

        fin_grupo = time.time()
        print(f"→ {round(fin_grupo-inicio_grupo,1)}s (acumulado: {round(fin_grupo-inicio_total,1)}s)")
        time.sleep(3)

    tiempo_total = round(time.time() - inicio_total, 1)
    print(f"\nTiempo total perfil {perfil}: {tiempo_total}s ({round(tiempo_total/60,1)} min)")
    return todas_señales

# Prueba con los 3 perfiles
resultados_perfiles = {}
inicio_global = time.time()

for perfil in ["conservador", "moderado", "agresivo"]:
    print(f"\nRANKING — Perfil {perfil.upper()}")
    print("="*70)
    señales = ranking_50(TICKERS, perfil=perfil)
    resultados_perfiles[perfil] = señales

    buys  = [s for s in señales if s.decision == "BUY"]
    holds = [s for s in señales if s.decision == "HOLD"]
    sells = [s for s in señales if s.decision == "SELL"]

    print(f"\nResumen: BUY={len(buys)} | HOLD={len(holds)} | SELL={len(sells)}")
    print(f"\n{'Ticker':<8} {'Empresa':<22} {'Señal':<6} {'Conf':<6}")
    print("-"*50)
    for s in sorted(señales, key=lambda x: x.confianza, reverse=True):
        nombre = NOMBRES.get(s.ticker, s.ticker)
        print(f"{s.ticker:<8} {nombre:<22} {s.decision:<6} {s.confianza}/10")

fin_global = time.time()
print(f"\nTiempo total los 3 perfiles: {round((fin_global-inicio_global)/60,1)} minutos")


RANKING — Perfil CONSERVADOR
Procesando grupo 1/10: ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META'] → 15.4s (acumulado: 15.4s)
Procesando grupo 2/10: ['AVGO', 'AMD', 'ORCL', 'INTC', 'QCOM'] → 15.8s (acumulado: 34.1s)
Procesando grupo 3/10: ['JNJ', 'UNH', 'LLY', 'ABBV', 'MRK'] → 16.6s (acumulado: 53.7s)
Procesando grupo 4/10: ['TMO', 'ABT', 'PFE', 'MDT', 'BMY'] → 18.0s (acumulado: 74.7s)
Procesando grupo 5/10: ['JPM', 'BAC', 'WFC', 'GS', 'MS'] → 14.5s (acumulado: 92.2s)
Procesando grupo 6/10: ['BLK', 'AXP', 'C', 'USB', 'PNC'] → 15.7s (acumulado: 110.9s)
Procesando grupo 7/10: ['AMZN', 'TSLA', 'HD', 'MCD', 'NKE'] → 14.4s (acumulado: 128.3s)
Procesando grupo 8/10: ['SBUX', 'TGT', 'LOW', 'F', 'GM'] → 19.8s (acumulado: 151.1s)
Procesando grupo 9/10: ['XOM', 'CVX', 'COP', 'SLB', 'EOG'] → 22.4s (acumulado: 176.6s)
Procesando grupo 10/10: ['CAT', 'BA', 'HON', 'GE', 'LMT'] → 19.2s (acumulado: 198.8s)

Tiempo total perfil conservador: 201.8s (3.4 min)

Resumen: BUY=7 | HOLD=43 | SELL=0

Ticker   Empr

In [24]:
import sys
sys.path.append("../src")
from sistema_llm_puro import cargar_datos, orquestador, ranking_50, get_llm, TICKERS, NOMBRES, PERFILES

llm   = get_llm()
datos = cargar_datos()

# Prueba rapida
r = orquestador("AAPL", datos, llm, perfil="moderado")
print(f"Decision: {r.decision} ({r.confianza}/10)")
print(f"Motivo  : {r.justificacion}")

Datos cargados: 50/50 tickers
Decision: BUY (7/10)
Motivo  : El perfil del inversor es moderado, por lo que acepta volatilidad moderada y busca un equilibrio entre rentabilidad y riesgo. En este caso, los agentes técnico y fundamental recomiendan comprar acciones de AAPL, lo que coincide con el perfil del inversor. El agente Riesko indica que AAPL es más volátil que el mercado en general, pero no demasiado, y genera valor adicional sobre el rendimiento del mercado. Sin embargo, hay un riesgo alto de pérdidas en comparación con otros activos. El agente Sentimiento recomienda comprar acciones de AAPL debido a la noticia 'Why Apple’s WWDC 2026 shows a material positive reset of criticized AI strategy?' que sugiere que Apple está revisando su estrategia de inteligencia artificial criticada y podría mejorar en este aspecto. La simulación Monte Carlo indica una probabilidad de subida del precio de AAPL superior al 60% en los próximos 20 días.


---

## Sistema Multiagente hibrido

La arquitectura hibrido sustituye los LLM individuales por modelos especializados más rápidos y reproducibles:

| Agente | LLM Puro | Sistema Hibrido |
|---|---|---|
| Técnico | Mistral 7B | Random Forest especializado |
| Fundamental | Mistral 7B | Random Forest especializado |
| Riesgo | Mistral 7B | Random Forest especializado |
| Sentimiento | Mistral 7B | **FinBERT** |
| Monte Carlo | Mistral 7B | Matemático puro (sin LLM) |
| Orquestador | Mistral 7B | Mistral 7B (único LLM) |

**Ventajas:**
- Solo 1 llamada LLM por acción en vez de 6
- RF y FinBERT son deterministas y reproducibles
- FinBERT especializado en texto financiero
- Mistral reservado para razonamiento de alto nivel


In [17]:
# Verificar dependencias de FinBERT
import subprocess, sys

def instalar(paquete):
    subprocess.check_call([sys.executable, "-m", "pip", "install", paquete, "-q"])

for paquete, nombre in [("transformers", "transformers"), ("torch", "torch"), ("joblib", "joblib")]:
    try:
        __import__(paquete)
        print(f"✓ {nombre} ya instalado")
    except ImportError:
        print(f"Instalando {nombre}...")
        instalar(paquete)

print("\n✓ Dependencias sistema hibrido listas")


✓ transformers ya instalado
✓ torch ya instalado
✓ joblib ya instalado

✓ Dependencias sistema hibrido listas


In [18]:
import sys
sys.path.append("../src")

from sistema_hibrido import (
    AgentesV3,
    cargar_datos,
    cargar_noticias,
    TICKERS, NOMBRES, PERFILES,
)

datos = cargar_datos()
cargar_noticias()

sistema = AgentesV3()


Datos cargados: 50/50 tickers
Noticias ya en memoria: 159,997 registros


In [19]:
# Entrenamiento de los 3 Random Forest especializados
# Carga desde PKL si ya existen en ../results/modelos_v3/
resultados_entrenamiento = sistema.entrenar(datos)

if resultados_entrenamiento:
    print("\n── Métricas de entrenamiento ──")
    for nombre, r in resultados_entrenamiento.items():
        print(f"\n{nombre.upper()}")
        print(f"  Acc train : {r['acc_train']:.3f}")
        print(f"  Acc test  : {r['acc_test']:.3f}")
        print(f"  Params    : {r['best_params']}")


Cargando modelos RF v3 desde disco...
Modelos cargados.


In [20]:
# Prueba del sistema hibrido con AAPL (tiempo real)
import time

print("Analizando AAPL con sistema hibrido...")
print("="*55)
print("(Solo Mistral hace una llamada LLM, el orquestador)")
print()

inicio = time.time()
resultado_v3 = sistema.orquestador("AAPL", datos, perfil="moderado")
elapsed = round(time.time() - inicio, 1)

print(f"Decision   : {resultado_v3.decision}")
print(f"Confianza  : {resultado_v3.confianza}/10")
print(f"Motivo     : {resultado_v3.justificacion}")
print(f"\nTiempo total: {elapsed}s")


Analizando AAPL con sistema hibrido...
(Solo Mistral hace una llamada LLM, el orquestador)

Cargando FinBERT (primera vez, puede tardar ~30s)...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 7473.23it/s]


FinBERT cargado.
Decision   : HOLD
Confianza  : 9/10
Motivo     : El perfil del inversor es moderado y acepta volatilidad moderada. Los indicadores actuales indican que la tecnología, fondos, riesgo y sentimientos están en equilibrio. Es mejor mantener la posición actual hasta que los indicadores cambien significativamente.

Tiempo total: 10.0s


In [7]:
# Comparativa de los 3 perfiles con AAPL — v3
print("COMPARATIVA DE PERFILES sistema hibrido — AAPL")
print("="*55)

for perfil in ["conservador", "moderado", "agresivo"]:
    r = sistema.orquestador("AAPL", datos, perfil=perfil)
    print(f"\nPerfil {perfil.upper()}:")
    print(f"  Decision  : {r.decision}")
    print(f"  Confianza : {r.confianza}/10")
    print(f"  Motivo    : {r.justificacion}")


COMPARATIVA DE PERFILES v3 — AAPL

Perfil CONSERVADOR:
  Decision  : HOLD
  Confianza : 9/10
  Motivo    : El perfil del inversor es conservador y prefiere preservar su capital sobre la rentabilidad. La señal técnica indica HOLD, lo que significa que no hay una clara oportunidad de compra en este momento.

Perfil MODERADO:
  Decision  : HOLD
  Confianza : 9/10
  Motivo    : El perfil del inversor es moderado y acepta volatilidad moderada. Los indicadores actuales indican que la tecnología, fondos, riesgo y sentimientos están en equilibrio. Es mejor mantenerse en posición HOLD hasta que los indicadores se vuelvan más positivos.

Perfil AGRESIVO:
  Decision  : BUY
  Confianza : 9/10
  Motivo    : El perfil agresivo del inversor busca maximizar la rentabilidad aceptando alto riesgo. Las señales actuales indican una tendencia positiva en el mercado tecnológico y de fondos.


In [21]:
# Ranking de las 50 acciones con v3 (perfil moderado)
import time

print("RANKING 50 ACCIONES — Sistema hibrido — Perfil MODERADO")
print("="*60)

inicio = time.time()
señales_v3 = []

for ticker in TICKERS:
    if ticker not in datos:
        continue
    try:
        r = sistema.orquestador(ticker, datos, perfil="moderado")
        señales_v3.append(r)
        print(f"  {ticker:<6} → {r.decision} ({r.confianza}/10)")
    except Exception as e:
        print(f"  {ticker:<6} → ERROR: {e}")

elapsed = round(time.time() - inicio, 1)

buys  = [s for s in señales_v3 if s.decision == "BUY"]
holds = [s for s in señales_v3 if s.decision == "HOLD"]
sells = [s for s in señales_v3 if s.decision == "SELL"]

print(f"\n{'='*60}")
print(f"Resumen v3 moderado: BUY={len(buys)} | HOLD={len(holds)} | SELL={len(sells)}")
print(f"Tiempo total: {elapsed}s ({round(elapsed/60,1)} min)")
print()

# Top 15 por confianza — filtramos tickers validos (solo mayusculas y letras)
señales_validas = [s for s in señales_v3 if s.ticker == s.ticker.upper() and s.ticker in TICKERS]
print(f"{'Ticker':<8} {'Empresa':<22} {'Señal':<6} {'Conf'}")
print("-"*50)
for s in sorted(señales_validas, key=lambda x: x.confianza, reverse=True)[:15]:
    nombre = NOMBRES.get(s.ticker, s.ticker)
    print(f"{s.ticker:<8} {nombre:<22} {s.decision:<6} {s.confianza}/10")


RANKING 50 ACCIONES — Sistema hibrido — Perfil MODERADO
  AAPL   → HOLD (8/10)
  MSFT   → HOLD (9/10)
  NVDA   → BUY (7/10)
  GOOGL  → HOLD (9/10)
  META   → HOLD (9/10)
  AVGO   → BUY (8/10)
  AMD    → BUY (8/10)
  ORCL   → BUY (8/10)
  INTC   → BUY (8/10)
  QCOM   → BUY (9/10)
  JNJ    → HOLD (9/10)
  UNH    → BUY (8/10)
  LLY    → BUY (8/10)
  ABBV   → BUY (7/10)
  MRK    → BUY (8/10)
  TMO    → BUY (7/10)
  ABT    → HOLD (9/10)
  PFE    → HOLD (9/10)
  MDT    → HOLD (8/10)
  BMY    → HOLD (9/10)
  JPM    → HOLD (9/10)
  BAC    → HOLD (9/10)
  WFC    → BUY (9/10)
  GS     → BUY (7/10)
  MS     → BUY (5/10)
  BLK    → HOLD (8/10)
  AXP    → HOLD (9/10)
  C      → BUY (8/10)
  USB    → BUY (7/10)
  PNC    → BUY (8/10)
  AMZN   → HOLD (9/10)
  TSLA   → BUY (7/10)
  HD     → HOLD (9/10)
  MCD    → HOLD (9/10)
  NKE    → BUY (7/10)
  SBUX   → HOLD (9/10)
  TGT    → BUY (7/10)
  LOW    → HOLD (9/10)
  F      → BUY (9/10)
  GM     → BUY (9/10)
  XOM    → HOLD (9/10)
  CVX    → HOLD (9/10)


In [23]:
# Comparativa LLM vs Hibrido
print("COMPARATIVA ARQUITECTURA LLM puro vs Hibrido")
print("="*55)

if señales_v3:
    buys_v3  = sum(1 for s in señales_v3 if s.decision == "BUY")
    holds_v3 = sum(1 for s in señales_v3 if s.decision == "HOLD")
    sells_v3 = sum(1 for s in señales_v3 if s.decision == "SELL")

    print(f"{'':25} {'LLM (Mistral x6)':>18} {'Sistema hibrido (RF+FinBERT)':>18}")
    print("-"*62)
    print(f"{'BUY':25} {'~25':>18} {buys_v3:>18}")
    print(f"{'HOLD':25} {'~25':>18} {holds_v3:>18}")
    print(f"{'SELL':25} {'~0':>18} {sells_v3:>18}")
    print(f"{'Llamadas LLM/ranking':25} {'50 × 6 = 300':>18} {'50 × 1 = 50':>18}")
    print(f"{'Tiempo/ranking':25} {'~8-9 min':>18} {f'~{round(elapsed/60,1)} min':>18}")
    print(f"{'Determinista':25} {'No':>18} {'Sí (RF+FinBERT)':>18}")


COMPARATIVA ARQUITECTURA LLM puro vs Hibrido
                            LLM (Mistral x6) Sistema hibrido (RF+FinBERT)
--------------------------------------------------------------
BUY                                      ~25                 26
HOLD                                     ~25                 24
SELL                                      ~0                  0
Llamadas LLM/ranking            50 × 6 = 300        50 × 1 = 50
Tiempo/ranking                      ~8-9 min           ~3.7 min
Determinista                              No    Sí (RF+FinBERT)
